# Snowflake Connector for ServiceNow - OAuth Client Credentials Setup

**Prerequisites:**
1. Install the connector from **Marketplace > search "Snowflake Connector for ServiceNow" > Get** (this step must be done via the UI)
2. In ServiceNow: enable `glide.oauth.inbound.client.credential.grant_type.enabled` = `true` (via `sys_properties.list`)
3. In ServiceNow: **System OAuth > Application Registry > New > Create an OAuth API endpoint for external clients**
4. Note the **Client ID** and **Client Secret** from ServiceNow

Once the connector is installed and you have your OAuth credentials, run the cells below **one at a time**.

In [ ]:
%%sql -r variables_set
SET application_name = 'SNOWFLAKE_CONNECTOR_FOR_SERVICENOW';
SET connector_warehouse = 'SNOWFLAKE_CONNECTOR_FOR_SERVICENOW_WAREHOUSE';
SET servicenow_instance_domain = '<servicenow_instance>.service-now.com';

SET destination_database = 'SNOWFLAKE_CONNECTOR_FOR_SERVICENOW_DEST_DB';
SET destination_schema = 'SNOWFLAKE_CONNECTOR_FOR_SERVICENOW_DEST_SCHEMA';

SET secret_database = 'CONNECTORS_SECRET';
SET secret_schema = 'SNOWFLAKE_CONNECTOR_FOR_SERVICENOW';
SET secret_database_schema = $secret_database || '.' || $secret_schema;
SET secret_fqn = $secret_database_schema || '.' || 'SECRET';

SET network_rule_fqn = $secret_database_schema || '.' || 'NETWORK_RULE';
SET external_access_integration_name = 'SNOWFLAKE_CONNECTOR_FOR_SERVICENOW_EXTERNAL_ACCESS_INTEGRATION';
SET security_integration_name = 'SNOWFLAKE_CONNECTOR_FOR_SERVICENOW_SECURITY_INTEGRATION';

SET destination_database_schema = $destination_database || '.' || $destination_schema;
SET servicenow_instance_url = 'https://' || $servicenow_instance_domain || '/';
SET oauth_token_endpoint = $servicenow_instance_url || 'oauth_token.do';

In [ ]:
%%sql -r essential_objects
USE ROLE ACCOUNTADMIN;

CREATE WAREHOUSE IF NOT EXISTS IDENTIFIER($connector_warehouse)
   WAREHOUSE_SIZE = 'MEDIUM'
   AUTO_RESUME = TRUE;
CREATE DATABASE IF NOT EXISTS IDENTIFIER($secret_database);
CREATE SCHEMA IF NOT EXISTS IDENTIFIER($secret_database_schema);
CREATE DATABASE IF NOT EXISTS IDENTIFIER($destination_database);
CREATE SCHEMA IF NOT EXISTS IDENTIFIER($destination_database_schema);

In [ ]:
%%sql -r security_integration
CREATE SECURITY INTEGRATION IDENTIFIER($security_integration_name)
   TYPE = API_AUTHENTICATION
   AUTH_TYPE = OAUTH2
   OAUTH_CLIENT_AUTH_METHOD = CLIENT_SECRET_POST
   OAUTH_CLIENT_ID = '<client_id>'
   OAUTH_CLIENT_SECRET = '<client_secret>'
   OAUTH_TOKEN_ENDPOINT = $oauth_token_endpoint
   OAUTH_GRANT = 'CLIENT_CREDENTIALS'
   OAUTH_ALLOWED_SCOPES = ('useraccount')
   ENABLED = TRUE;

In [ ]:
%%sql -r secret_created
CREATE SECRET IDENTIFIER($secret_fqn)
   TYPE = OAUTH2
   API_AUTHENTICATION = $security_integration_name
   OAUTH_SCOPES = ('useraccount');

In [ ]:
%%sql -r network_rule
CREATE NETWORK RULE IDENTIFIER($network_rule_fqn)
   MODE = 'EGRESS'
   TYPE = 'HOST_PORT'
   VALUE_LIST = ($servicenow_instance_domain);

In [ ]:
%%sql -r eai_created
CREATE PROCEDURE execute_immediate_create_ea_integration()
RETURNS VARIANT
EXECUTE AS CALLER
AS
BEGIN
   EXECUTE IMMEDIATE '
      CREATE EXTERNAL ACCESS INTEGRATION IDENTIFIER($external_access_integration_name)
      ALLOWED_NETWORK_RULES = ($network_rule_fqn)
      ALLOWED_AUTHENTICATION_SECRETS = ('  ||  $secret_fqn  ||  ') ENABLED = TRUE
   ';
END;

CALL execute_immediate_create_ea_integration();
DROP PROCEDURE IF EXISTS execute_immediate_create_ea_integration();

In [ ]:
%%sql -r grants_applied
GRANT EXECUTE TASK ON ACCOUNT TO APPLICATION IDENTIFIER($application_name);
GRANT EXECUTE MANAGED TASK ON ACCOUNT TO APPLICATION IDENTIFIER($application_name);

GRANT USAGE ON WAREHOUSE IDENTIFIER($connector_warehouse) TO APPLICATION IDENTIFIER($application_name);
GRANT USAGE ON DATABASE IDENTIFIER($destination_database) TO APPLICATION IDENTIFIER($application_name);
GRANT USAGE ON SCHEMA IDENTIFIER($destination_database_schema) TO APPLICATION IDENTIFIER($application_name);

GRANT CREATE TABLE ON SCHEMA IDENTIFIER($destination_database_schema) TO APPLICATION IDENTIFIER($application_name);
GRANT CREATE VIEW ON SCHEMA IDENTIFIER($destination_database_schema) TO APPLICATION IDENTIFIER($application_name);

GRANT USAGE ON INTEGRATION IDENTIFIER($external_access_integration_name) TO APPLICATION IDENTIFIER($application_name);
GRANT USAGE ON DATABASE IDENTIFIER($secret_database) TO APPLICATION IDENTIFIER($application_name);
GRANT USAGE ON SCHEMA IDENTIFIER($secret_database_schema) TO APPLICATION IDENTIFIER($application_name);
GRANT READ ON SECRET IDENTIFIER($secret_fqn) TO APPLICATION IDENTIFIER($application_name);

In [ ]:
%%sql -r ownership_granted
CALL SYSTEM$GRANT_OWNERSHIP_TO_APPLICATION($application_name, true, $destination_database, $destination_schema);

In [ ]:
%%sql -r connector_configured
USE APPLICATION IDENTIFIER($application_name);

CALL CONFIGURE_CONNECTOR({
   'warehouse': $connector_warehouse,
   'destination_database': $destination_database,
   'destination_schema': $destination_schema
});

In [ ]:
%%sql -r connection_configured
CALL SET_CONNECTION_CONFIGURATION({
   'service_now_url': $servicenow_instance_url,
   'secret': $secret_fqn,
   'external_access_integration': $external_access_integration_name
});

In [ ]:
%%sql -r finalized
CALL FINALIZE_CONNECTOR_CONFIGURATION({
   'journal_table': 'sys_audit_delete'
});